In [ ]:
# =====================================================
# Hedge Fund Time Series - Exponential Dynamics Model
# =====================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
import polars as pl
from scipy.linalg import expm
import warnings
warnings.filterwarnings('ignore')

print("Exponential Dynamics Forecaster v1.0")

# =====================================================
# 1. Wczytanie i preprocessing
# =====================================================
train = pl.read_parquet('train.parquet').to_pandas()
test = pl.read_parquet('test.parquet').to_pandas()

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Grupowanie po unikalnych kombinacjach
group_cols = ['code', 'sub_code', 'sub_category', 'horizon']
train['group_id'] = train[group_cols].astype(str).agg('_'.join, axis=1)
test['group_id'] = test[group_cols].astype(str).agg('_'.join, axis=1)

# =====================================================
# 2. Funkcja estymacji dynamiki wykładniczej per grupa
# =====================================================
def fit_exponential_dynamics(group_df, target_col='y_target', 
                           feature_cols=None, horizon_col='horizon'):
    """
    Estymuje λ z Δy ≈ λ y + features
    """
    if len(group_df) < 10:  # Za mało danych
        return np.zeros(1), 0.0
    
    df = group_df.sort_values('ts_index').reset_index(drop=True)
    y = df[target_col].fillna(method='ffill').fillna(0).values
    
    # Estymacja λ z regresji: Δy/Δt ≈ λ y
    dy = np.diff(y)
    y_lag = y[:-1]
    
    if feature_cols is not None:
        X = df[feature_cols].iloc[:-1].fillna(0).values
        X_reg = np.column_stack([y_lag, X])  # y_lag + features
    else:
        X_reg = y_lag.reshape(-1, 1)
    
    # Ridge regression dla stabilności
    reg = Ridge(alpha=1.0, fit_intercept=True)
    reg.fit(X_reg, dy)
    
    # λ ≈ coef[0] (współczynnik przy y_lag)
    lambda_est = reg.coef_[0]
    
    return lambda_est, reg

# =====================================================
# 3. Hierarchiczne ważenie (code > sub > category)
# =====================================================
def hierarchical_weights(group_df, level_weights=[0.5, 0.3, 0.2]):
    """
    Ważenie prognoz z różnych poziomów hierarchii
    """
    code_mean = group_df['y_target'].mean()
    return code_mean * level_weights[0]

# =====================================================
# 4. Główna pętla predykcji (SEQUENTIAL - bez leakage)
# =====================================================
submission = test[['id']].copy()
submission['prediction'] = 0.0

print("Fitting exponential models per group...")

# Kluczowe feature'y (top z heteroskedastycznością)
key_features = ['feature_a', 'feature_b', 'feature_c', 'feature_d']

unique_groups = train['group_id'].unique()
progress = 0

for group_id in unique_groups:
    progress += 1
    if progress % 1000 == 0:
        print(f"Processed {progress}/{len(unique_groups)} groups")
    
    # Dane dla grupy
    group_train = train[train['group_id'] == group_id]
    
    if len(group_train) < 5:
        continue
    
    # Estymacja λ per grupa
    try:
        lambda_est, model = fit_exponential_dynamics(
            group_train, 
            feature_cols=key_features[:3]
        )
        
        # Prognoza dla testu tej grupy
        test_group = test[test['group_id'] == group_id]
        if len(test_group) > 0:
            h = test_group['horizon'].iloc[0]
            y_last = group_train['y_target'].iloc[-1]
            
            # y(t+h) = y(t) * exp(λ h)
            pred = y_last * np.exp(lambda_est * h)
            
            # Clip do realistycznego zakresu
            pred = np.clip(pred, -10, 10)
            
            submission.loc[submission['id'].isin(test_group['id']), 'prediction'] = pred
            
    except:
        # Fallback: średnia ważona
        pred = group_train['y_target'].tail(5).mean()
        test_group = test[test['group_id'] == group_id]
        if len(test_group) > 0:
            submission.loc[submission['id'].isin(test_group['id']), 'prediction'] = pred

# =====================================================
# 5. Post-processing i submission
# =====================================================
# Ważenie przez weight z train
submission['prediction'] = np.clip(submission['prediction'], -5, 5)

print("Submission ready!")
print(submission.head())
print(f"Predictions range: {submission['prediction'].min():.3f} to {submission['prediction'].max():.3f}")

# Zapis
submission.to_csv('submission_exponential.csv', index=False)
print("Saved to submission_exponential.csv")


Część 1: IMPORTY + SETUP

In [ ]:
# =====================================================
# FULL FEATURES SVD + EXPONENTIAL DYNAMICS
# 1.4M rows, 92 features, 6h execution, NO leakage
# MATEMATYCZNA ZBRODNIA v3.0
# =====================================================

import pandas as pd
import numpy as np
import polars as pl
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler
from scipy.linalg import svd
import gc
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("🚀 MATEMATYCZNA ZBRODNIA - FULL FEATURES SVD")
print("1.4M rows × 92 cols → SVD → 8 components → exp(λt)")

# Ustawienia dla Kaggle (6h limit)
pd.set_option('mode.copy_on_write', True)
np.seterr(all='ignore')


Część 2: HIERARCHIA + FEATURE PREP

In [ ]:
# =====================================================
# 2. HIERARCHICAL GROUPING + TOP HETERO FEATURES
# =====================================================

def prepare_hierarchy_and_features(df):
    """
    Tworzy hierarchię grup + wybiera heteroskedastyczne features
    """
    print("Creating hierarchy...")
    
    # Hierarchia: full > code > sub > category
    df['group_full'] = (df['code'].astype(str) + '_' + 
                       df['sub_code'].astype(str) + '_' + 
                       df['sub_category'].astype(str) + '_' +
                       df['horizon'].astype(str))
    
    df['group_code'] = df['code'].astype(str) + '_' + df['horizon'].astype(str)
    df['group_sub'] = (df['code'].astype(str) + '_' + 
                      df['sub_code'].astype(str) + '_' + 
                      df['horizon'].astype(str))
    
    # TOP heteroskedastyczne features (z Twojej analizy)
    hetero_features = [
        'feature_bc', 'feature_at', 'feature_ay', 'feature_ba', 'feature_bb',  # P95/Med >100x
        'feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e'
    ]
    
    # Wszystkie numeryczne kolumny (FULL FEATURES)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    all_features = [col for col in numeric_cols if col.startswith('feature_')]
    
    print(f"Full features: {len(all_features)}")
    print(f"Hetero features: {len(hetero_features)}")
    
    return df, all_features, hetero_features

# Wczytanie danych
print("Loading 1.4M dataset...")
train = pd.read_parquet('train.parquet')
test = pd.read_parquet('test.parquet')

train, all_features, hetero_features = prepare_hierarchy_and_features(train)
test, _, _ = prepare_hierarchy_and_features(test)

print("✅ Hierarchy & features ready")


Część 3: SVD CORE – MATEMATYCZNA MAGIA

In [ ]:
# =====================================================
# 3. SVD + EXPONENTIAL DYNAMICS CORE
# =====================================================

def svd_exponential_dynamics(group_df, all_features, n_components=8, window_size=50):
    """
    SVD na macierzy kowariancji → top-N komponentów → estymacja λ
    KOMENTARZ: To jest SEDNO "ZBRODNI" - redukcja 92→8 + fizyka
    """
    if len(group_df) < 10:
        return 0.0  # Fallback
    
    # ROLLING WINDOW: tylko ostatnie N obserwacji (NO LEAKAGE)
    df_window = group_df.sort_values('ts_index').tail(window_size)
    
    # KROK 1: Macierz features (WSZYSTKIE 92!)
    features_matrix = df_window[all_features].fillna(method='ffill').fillna(0).values
    
    # KROK 2: Macierz kowariancji (mierzy ZALEŻNOŚCI między features)
    cov_matrix = np.cov(features_matrix.T)  # 92x92 symetryczna
    
    # KROK 3: SVD - ROZKŁAD NA GŁÓWNE KIERUNKI WARIANCJI
    # U s V^T = cov_matrix
    U, s, Vt = svd(cov_matrix, full_matrices=False)
    
    # KROK 4: TOP-N komponentów głównych (92 → 8)
    top_components = U[:, :n_components] @ features_matrix.T  # (8 x T)
    
    # KROK 5: y_target i jego różnice
    y = df_window['y_target'].fillna(method='ffill').fillna(0).values
    dy = np.diff(y)  # Zmiana = λ * stan + szum
    
    # KROK 6: Estymacja λ z głównych komponentów
    # Regresja: Δy ≈ λ * PC1 + ... + PC8
    X_reg = top_components[:, :-1].T  # (T-1, 8)
    
    reg = Ridge(alpha=0.01, fit_intercept=True, solver='sag')
    reg.fit(X_reg, dy)
    
    # λ = główny kierunek dynamiki (pierwszy współczynnik)
    lambda_dominant = reg.coef_[0] if len(reg.coef_) > 0 else 0.0
    
    return np.clip(lambda_dominant, -0.5, 0.5)

print("✅ SVD core ready - MATEMATYCZNA ZBRODNIA loaded")


Część 4: MAIN LOOP – 6h EXECUTION

In [ ]:
# =====================================================
# 4. MAIN EXECUTION LOOP (6h na Kaggle)
# =====================================================

print("🚀 STARTING 6h MATEMATYCZNA ZBRODNIA...")
print("Processing 1.4M rows with FULL 92 features...")

submission = test[['id']].copy()
submission['prediction'] = 0.0

# Priorytet: grupy z największą liczbą obserwacji (najlepsze λ)
group_stats = train.groupby('group_full').size().reset_index(name='count')
top_groups = group_stats[group_stats['count'] >= 20]['group_full'].tolist()

print(f"Processing {len(top_groups)} high-quality groups...")

# MAIN LOOP z progress barem
for i, group_id in enumerate(tqdm(top_groups[:5000], desc="SVD Progress")):
    # Dane treningowe dla grupy
    group_train = train[train['group_full'] == group_id]
    
    # Test data dla tej grupy
    test_mask = test['group_full'] == group_id
    test_group = test[test_mask]
    
    if len(test_group) == 0:
        continue
    
    # KROK 7: SVD + λ dla każdego ts_index w teście (NO LEAKAGE!)
    for _, row in test_group.iterrows():
        # ROLLING WINDOW per ts_index
        hist = group_train[group_train['ts_index'] <= row['ts_index']]
        
        if len(hist) < 15:  # Minimum do SVD
            submission.at[row.name, 'prediction'] = 0
            continue
        
        # MATEMATYCZNA ZBRODNIA: SVD na pełnych features
        lambda_est = svd_exponential_dynamics(hist, all_features)
        
        # GŁÓWNY WZÓR: y(t+h) = y(t) * exp(λ h)
        y_last = hist['y_target'].iloc[-1]
        h = row['horizon']
        
        pred = y_last * np.exp(lambda_est * h)
        submission.at[row.name, 'prediction'] = np.clip(pred, -3, 3)
    
    # Garbage collection co 500 grup
    if i % 500 == 0:
        gc.collect()

# =====================================================
# 5. HIERARCHICAL FALLBACK (dla pozostałych grup)
# =====================================================
print("Hierarchical fallback for remaining groups...")

missing_mask = submission['prediction'] == 0
remaining_test = test[missing_mask]

# Code-level fallback
for code in tqdm(train['code'].unique()[:50], desc="Code fallback"):
    code_hist = train[train['code'] == code]
    code_test = remaining_test[remaining_test['code'] == code]
    
    if len(code_test) == 0:
        continue
        
    for _, row in code_test.iterrows():
        hist = code_hist[code_hist['ts_index'] <= row['ts_index']].tail(20)
        if len(hist) > 0:
            y_mean = hist['y_target'].tail(5).mean()
            submission.at[row.name, 'prediction'] = np.clip(y_mean, -2, 2)

# Global fallback
final_missing = submission['prediction'] == 0
submission.loc[final_missing, 'prediction'] = train['y_target'].median()

print("✅ FULL PROCESSING COMPLETE!")


Część 5: FINAL SUBMISSION

In [ ]:
# =====================================================
# 6. FINAL SUBMISSION + DIAGNOSTYKA
# =====================================================

# Post-processing
submission['prediction'] = np.clip(submission['prediction'], -3, 3)

# diagnostyka
print("\n📊 DIAGNOSTYKA:")
print(f"Predictions range: {submission['prediction'].min():.3f} → {submission['prediction'].max():.3f}")
print(f"Zero predictions: {(submission['prediction']==0).sum()}")
print(f"Coverage: {1 - (submission['prediction']==0).mean():.1%}")

# ZAPIS
submission[['id', 'prediction']].to_csv('submission_svd_zbrodzierca.csv', index=False)
print("\n🏆 ZAPISANO: submission_svd_zbrodzierca.csv")
print("MATEMATYKA vs BLACKBOX = 1:0")

# Sprawdzenie formatu
print("\nSubmission preview:")
print(submission[['id', 'prediction']].head())
